# 11: DSP Blocks and Correlators

## Objective
Use FIR filters, DDS tuning, and hardware correlators for advanced signal processing.

In [ ]:
BITSTREAM_PATH = "/path/to/firmware.bit"
USE_PROXY = False
PROXY_IP = "192.168.1.100"

from qick import QickSoc, QickConfig
import numpy as np
import matplotlib.pyplot as plt

if USE_PROXY:
    from qick.pyro import make_proxy
    soc = make_proxy(PROXY_IP)
else:
    soc = QickSoc(bitfile=BITSTREAM_PATH)

print("Ready")

In [ ]:
# FIR filter coefficients (low-pass)
fir_coeffs = [100, 200, 300, 400, 500, 400, 300, 200, 100]
fir_coeffs = [int(c) for c in np.array(fir_coeffs) / np.sum(fir_coeffs) * 32767]

config = QickConfig()
config.set_fir_filter(ch=0, coeffs=fir_coeffs)
config.pulse(ch=0, style="const", length=1000, gain=0.5)

soc.run(config)
print("FIR filter applied to DAC 0")
print(f"Coefficients: {fir_coeffs}")

In [ ]:
# DDS tuning during sequence
config = QickConfig(reps=1)

config.set_dds_freq(ch=0, freq=100e6, t=0)
config.pulse(ch=0, style="const", length=500, gain=0.5)
config.set_dds_freq(ch=0, freq=120e6, t=500)  # Change frequency mid-sequence
config.pulse(ch=0, style="const", length=500, gain=0.5)

soc.run(config)
print("DDS frequency changed at t=500 ns")

In [ ]:
# Hardware correlator for two channels
config = QickConfig(reps=1000, soft_avgs=1)

config.pulse(ch=0, style="const", length=100, gain=0.5)
config.pulse(ch=1, style="const", length=100, gain=0.5)

config.enable_correlator(ch0=0, ch1=1, length=100)

soc.run(config)
correlation = soc.get_correlation()

plt.figure()
plt.plot(np.real(correlation), label='Real')
plt.plot(np.imag(correlation), label='Imag')
plt.legend()
plt.title("Hardware Correlation: Ch0 vs Ch1")
plt.xlabel("Lag (samples)")
plt.show()

In [ ]:
# Multi-tone generation using DDS
config = QickConfig(reps=1)

config.gen_channel(ch=0, freq=100e6, gain=0.3, style="const")
config.gen_channel(ch=0, freq=200e6, gain=0.3, style="const", sum=True)  # Sum with existing

soc.run(config)
print("Multi-tone signal: 100 MHz + 200 MHz on DAC 0")